# Data Exploration - Epstein Accountability Index

Attribution: Scaffolded with AI assistance (Claude, Anthropic)

This notebook explores the raw Epstein case file data and performs initial analysis.

In [ ]:
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# Set plotting style
sns.set_style('darkgrid')
plt.rcParams['figure.figsize'] = (12, 6)

%matplotlib inline

## Load Raw Data

In [ ]:
# Load aggregated JSON files
raw_data_dir = Path('../data/raw')
json_files = list(raw_data_dir.glob('ds*_agg.json'))

print(f"Found {len(json_files)} JSON files")

# Load first file as example
if json_files:
    with open(json_files[0], 'r') as f:
        sample_data = json.load(f)
    print(f"\nSample file contains {len(sample_data)} documents")
    print(f"\nSample document structure:")
    first_doc = next(iter(sample_data.values()))
    for key in first_doc.keys():
        print(f"  - {key}: {type(first_doc[key]).__name__}")

## Document Statistics

In [ ]:
# Analyze document corpus
all_docs = []

for json_file in json_files:
    with open(json_file, 'r') as f:
        data = json.load(f)
        for doc_id, doc_data in data.items():
            all_docs.append({
                'doc_id': doc_id,
                'pages': doc_data.get('pages', 0),
                'success': doc_data.get('success', False),
                'text_length': len(doc_data.get('text', '')),
                'status_reason': doc_data.get('status_reason', '')
            })

docs_df = pd.DataFrame(all_docs)
print(f"Total documents: {len(docs_df)}")
print(f"Successful extractions: {docs_df['success'].sum()}")
print(f"\nDocument statistics:")
print(docs_df.describe())

In [ ]:
# Visualize document statistics
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Page count distribution
axes[0].hist(docs_df['pages'], bins=50, edgecolor='black')
axes[0].set_xlabel('Number of Pages')
axes[0].set_ylabel('Frequency')
axes[0].set_title('Document Page Count Distribution')

# Text length distribution
axes[1].hist(docs_df['text_length'], bins=50, edgecolor='black')
axes[1].set_xlabel('Text Length (characters)')
axes[1].set_ylabel('Frequency')
axes[1].set_title('Document Text Length Distribution')

plt.tight_layout()
plt.show()

## Load Processed Data

In [ ]:
# Load severity scores
severity_df = pd.read_csv('../data/processed/severity_scores.csv')
print(f"Severity scores: {len(severity_df)} people")
print(severity_df.head())

In [ ]:
# Load consequences
consequences_df = pd.read_csv('../data/processed/consequences.csv')
print(f"\nConsequences: {len(consequences_df)} people")
print(consequences_df.head())

In [ ]:
# Visualize consequence tier distribution
tier_counts = consequences_df['consequence_tier'].value_counts().sort_index()

plt.figure(figsize=(8, 5))
tier_counts.plot(kind='bar', color=['green', 'orange', 'red'])
plt.xlabel('Consequence Tier')
plt.ylabel('Number of People')
plt.title('Consequence Tier Distribution')
plt.xticks(rotation=0)
plt.xticks([0, 1, 2], ['None', 'Soft', 'Hard'])
plt.tight_layout()
plt.show()

## Initial Correlation Analysis

In [ ]:
# Merge severity and consequences
merged_df = severity_df.merge(consequences_df[['name', 'consequence_tier']], on='name', how='inner')

# Compute correlation
correlation = merged_df['severity_score'].corr(merged_df['consequence_tier'])
print(f"Correlation between severity and consequences: {correlation:.3f}")

# Scatter plot
plt.figure(figsize=(10, 6))
plt.scatter(merged_df['severity_score'], merged_df['consequence_tier'], alpha=0.6)
plt.xlabel('Severity Score')
plt.ylabel('Consequence Tier')
plt.title('Severity vs. Consequences')
plt.yticks([0, 1, 2], ['None', 'Soft', 'Hard'])
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()